# Does Concentration-Dependent Receptor Encoding Predict Behavioral Response?**Justine Sherry — Capstone Thesis (W2026)**This notebook tests whether **variability in olfactory receptor population activation across concentrations** predicts **variability in behavioral response** to the same odorants.We combine two datasets:1. **Manoel et al. (2021)** — Spontaneous behavioral responses of mice to 12 odorants at three concentrations (8.5 µM, 850 µM, 85 mM)2. **Mainland et al. (2015)** — Dose-response curves for ~300 human olfactory receptors to ~90 odorants**Hypothesis:** Concentration-dependent variability in olfactory receptor encoding predicts variability in behavioral response.**Specific Aim 1:** Characterize the concentration dependence of spontaneous, odor-evoked behavioral responses — organizing behaviors into innate vs. learned categories and quantifying how behavioral profiles change with concentration.**Specific Aim 2:** Characterize the concentration dependence of a virtual receptor population to the same odorants, and examine the relationship between sensory and behavioral responses.---**References:**- Manoel et al. (2021). *Deconstructing the mouse olfactory percept through an ethological atlas.* Current Biology, 31(11), 2521-2534.- Mainland et al. (2015). *The missense of smell: functional variability in the human odorant receptor repertoire.* Nature Neuroscience, 18(1), 114-120.

--- ## 0. Setup

In [ ]:
import subprocess, sysfor pkg in ['statsmodels', 'openpyxl']:    try:        __import__(pkg)    except ImportError:        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])        print(f"Installed {pkg}")

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import statsfrom scipy.spatial.distance import pdist, squareform, euclideanfrom scipy.cluster.hierarchy import linkage, dendrogram, leaves_listfrom scipy.optimize import curve_fitfrom sklearn.preprocessing import MinMaxScalerfrom sklearn.decomposition import NMFimport warningswarnings.filterwarnings('ignore')sns.set_theme(style='whitegrid', context='notebook')%matplotlib inline

--- ## 1. Load the datasetsAll data is hosted publicly on GitHub — just run the cells.

In [ ]:
# --- Manoel et al. (2021): Mouse behavioral responses at 3 concentrations ---BASE = "https://raw.githubusercontent.com/CastroLab/castro-teach-codebase/main/capstone_projects"import openpyxlfrom io import BytesIOimport urllib.request# Download the Excel file (can't read .xlsx directly from URL with pandas sometimes)xlsx_url = f"{BASE}/manoel/data/DataS1.xlsx"response = urllib.request.urlopen(xlsx_url)xlsx_data = BytesIO(response.read())conc_raw = pd.read_excel(xlsx_data,                         sheet_name='raw behavioral data (Fig S5A)',                         header=1, index_col=0)conc_raw.index.name = 'condition'conc_raw = conc_raw.iloc[:, :18]conc_raw.columns = conc_raw.columns.str.strip()conc_raw = conc_raw.dropna()# Parse condition names into odorant + concentrationdef parse_condition(cond):    cond = str(cond).strip()    if cond == 'H2O':        return 'H2O', 'control'    parts = cond.rsplit(' ', 1)    if len(parts) == 2 and parts[1] in ('8.5uM', '850uM', '85mM'):        return parts[0].strip(), parts[1]    return cond, 'unknown'parsed = [parse_condition(c) for c in conc_raw.index]conc_raw['odorant'] = [p[0] for p in parsed]conc_raw['concentration'] = [p[1] for p in parsed]# Behavioral columnsbehav_cols = [c for c in conc_raw.columns if c not in ('odorant', 'concentration')]# Separate water controlsdf_all = conc_raw.copy()df = conc_raw[conc_raw['odorant'] != 'H2O'].copy()CONC_ORDER = ['8.5uM', '850uM', '85mM']print(f"Loaded Manoel behavioral data: {len(df)} observations")print(f"  {df['odorant'].nunique()} odorants × {len(CONC_ORDER)} concentrations")print(f"  {len(behav_cols)} behavioral measures: {behav_cols}")

In [ ]:
# Manoel molecule metadata (names, CIDs)molecules = pd.read_csv(f"{BASE}/manoel/data/molecules.csv")# Abbreviation → full name mapping (from the paper)ABBREV_TO_NAME = {    '2HO': '2-heptanone',     'AMB': 'ambrettolid',    'DMP': '2,5-dimethylpyrazine', 'IAA': 'isoamylamine',    'IBT': '2-isobutylthiazole',   'IND': 'indole',    'PEA': '2-phenylethanol',       'PUT': 'putrescine',    'SKA': '3-methylindole',        'TMA': 'trimethylamine',    'TMT': 'TMT',                   'VAN': 'vanillin',}# Categorize behaviors as innate vs. learned/exploratory# Based on ethological interpretation from Manoel et al.INNATE_BEHAVIORS = [    'Freezing', 'Stretched attend', 'Jump/startle',    'Digging', 'Grooming',]LEARNED_BEHAVIORS = [    'Sniffing', 'Rearing', 'Walking',    'Feeding', 'Handling',]# Map which behaviors we have in our datainnate_cols = [b for b in behav_cols if any(ib.lower() in b.lower() for ib in INNATE_BEHAVIORS)]learned_cols = [b for b in behav_cols if any(lb.lower() in b.lower() for lb in LEARNED_BEHAVIORS)]other_cols = [b for b in behav_cols if b not in innate_cols and b not in learned_cols]print(f"Innate/defensive behaviors ({len(innate_cols)}): {innate_cols}")print(f"Learned/exploratory behaviors ({len(learned_cols)}): {learned_cols}")print(f"Other/unclassified ({len(other_cols)}): {other_cols}")

In [ ]:
# --- Mainland et al. (2015): Dose-response curves for human ORs ---dr = pd.read_csv(f"{BASE}/mainland/data/DR.tsv", sep="\t", quotechar='"')odors = pd.read_csv(f"{BASE}/mainland/data/Odors.tsv", sep="\t")receptors = pd.read_csv(f"{BASE}/mainland/data/Receptors.tsv", sep="\t",                        usecols=["OR", "Gene"])# Build lookup dictsodor_names = dict(zip(odors["Odor"], odors["OdorName"]))receptor_genes = dict(zip(receptors["OR"], receptors["Gene"]))# Drop vector controlsreceptors = receptors[~receptors["Gene"].str.contains("Vector", case=False, na=False)]real_or_ids = set(receptors["OR"])dr = dr[dr["OR"].isin(real_or_ids)]dr = dr.dropna(subset=["NormalizedLuc"])print(f"Loaded Mainland dose-response data:")print(f"  {len(dr):,} measurements across {dr['OR'].nunique()} receptors × {dr['Odor'].nunique()} odorants")

---# PART I — Concentration-Dependent Behavioral Responses (Manoel et al.)Manoel et al. measured spontaneous behavioral responses of mice presented with 12 odorants, each at three concentrations (8.5 µM, 850 µM, 85 mM). We first characterize how behavioral profiles depend on concentration, then organize behaviors into innate vs. learned categories.---## 2. Explore the behavioral data

In [ ]:
# Compute odorant × behavior means at each concentrationconc_odorant_means = {}odorants_12 = sorted(df['odorant'].unique())for conc in CONC_ORDER:    mask = df['concentration'] == conc    means = df[mask].groupby('odorant')[behav_cols].mean()    conc_odorant_means[conc] = means# Overview: heatmap of mean behavioral scores at mid concentrationref = conc_odorant_means['850uM']fig, ax = plt.subplots(figsize=(12, 6))sns.heatmap(ref, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,            linewidths=0.5, linecolor='white')ax.set_title('Mean Behavioral Scores at 850 µM (Mid Concentration)', fontsize=13, fontweight='bold')ax.set_ylabel('Odorant')plt.tight_layout()plt.show()print(f"\n12 odorants tested: {odorants_12}")

### 2.1 How do behavioral profiles change with concentration?We z-score each behavior (across odorants) within the mid-concentration panel, then apply the same ordering to low and high — so you can scan across and see what shifts.

In [ ]:
# Biclustered heatmaps: low / mid / high concentration# Z-score within mid-concentration for consistent scalingref_data = conc_odorant_means['850uM']z_mean = ref_data.mean()z_std = ref_data.std().replace(0, 1)# Bicluster on mid-concentrationfrom scipy.cluster.hierarchy import linkage, leaves_listfrom scipy.spatial.distance import pdistrow_link = linkage(pdist(ref_data.values), method='average')col_link = linkage(pdist(ref_data.values.T), method='average')row_order = leaves_list(row_link)col_order = leaves_list(col_link)ordered_odorants = ref_data.index[row_order]ordered_behaviors = ref_data.columns[col_order]fig, axes = plt.subplots(1, 3, figsize=(20, 6))for ax, conc, label in zip(axes, CONC_ORDER, ['LOW (8.5 µM)', 'MID (850 µM)', 'HIGH (85 mM)']):    data = conc_odorant_means[conc]    z_data = (data - z_mean) / z_std    z_data = z_data.loc[ordered_odorants, ordered_behaviors]        sns.heatmap(z_data, ax=ax, cmap='RdBu_r', center=0, vmin=-2, vmax=2,                xticklabels=True, yticklabels=(conc == '8.5uM'),                cbar=(conc == '85mM'), linewidths=0.3)    ax.set_title(label, fontsize=12, fontweight='bold')    ax.tick_params(axis='x', rotation=45)fig.suptitle('Behavioral Profiles Across Concentrations (z-scored to mid)',             fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.show()

### 2.2 Innate vs. learned behavioral responsesA key part of your thesis is organizing behaviors into **innate/defensive** (freezing, startle, stretched attend) vs. **learned/exploratory** (sniffing, rearing, walking, feeding). Let's see how these categories respond differently to concentration.

In [ ]:
# Compute category-level scores per odorant × concentrationcategory_scores = []for conc in CONC_ORDER:    mask = df['concentration'] == conc    for odorant in odorants_12:        omask = mask & (df['odorant'] == odorant)        subset = df[omask]        if len(subset) == 0:            continue                innate_score = subset[innate_cols].mean(axis=1).mean() if innate_cols else 0        learned_score = subset[learned_cols].mean(axis=1).mean() if learned_cols else 0                category_scores.append({            'odorant': odorant,            'concentration': conc,            'innate_mean': innate_score,            'learned_mean': learned_score,            'innate_learned_ratio': innate_score / learned_score if learned_score > 0 else np.nan,        })cat_df = pd.DataFrame(category_scores)# Plot: innate vs learned across concentrationsfig, axes = plt.subplots(1, 2, figsize=(14, 5))for ax, measure, title in zip(axes,    ['innate_mean', 'learned_mean'],    ['Innate/Defensive Behaviors', 'Learned/Exploratory Behaviors']):        for odorant in odorants_12:        odata = cat_df[cat_df['odorant'] == odorant]        ax.plot(range(3), odata[measure].values, 'o-', alpha=0.6, label=odorant)        ax.set_xticks(range(3))    ax.set_xticklabels(['8.5 µM', '850 µM', '85 mM'])    ax.set_xlabel('Concentration')    ax.set_ylabel('Mean Score')    ax.set_title(title, fontsize=12, fontweight='bold')axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)plt.tight_layout()plt.show()

---## 3. SA1 — Concentration dependence of behavioral responses### 3.1 Behavioral divergence: how much does each odorant's profile change with concentration?We compute the Euclidean distance between each odorant's behavioral profile at low vs. high concentration (in z-scored space). Odorants with large divergence are strongly concentration-dependent; those with small divergence behave similarly at all concentrations.

In [ ]:
# Euclidean distance between low and high concentration profiles (z-scored)all_means = pd.concat([conc_odorant_means[c] for c in CONC_ORDER])pooled_m = all_means.mean()pooled_s = all_means.std().replace(0, 1)divergences = []for odor in odorants_12:    low = conc_odorant_means['8.5uM'].loc[odor]    high = conc_odorant_means['85mM'].loc[odor]    low_z = (low - pooled_m) / pooled_s    high_z = (high - pooled_m) / pooled_s    d = np.linalg.norm(high_z.values - low_z.values)        # Also compute innate vs learned divergences separately    if innate_cols:        d_innate = np.linalg.norm(            ((high - pooled_m) / pooled_s)[innate_cols].values -            ((low - pooled_m) / pooled_s)[innate_cols].values)    else:        d_innate = np.nan    if learned_cols:        d_learned = np.linalg.norm(            ((high - pooled_m) / pooled_s)[learned_cols].values -            ((low - pooled_m) / pooled_s)[learned_cols].values)    else:        d_learned = np.nan        divergences.append({        'odorant': odor, 'full_name': ABBREV_TO_NAME.get(odor, odor),        'total_divergence': d,        'innate_divergence': d_innate,        'learned_divergence': d_learned,    })div_df = pd.DataFrame(divergences).sort_values('total_divergence', ascending=False)# Bar chartfig, ax = plt.subplots(figsize=(10, 5))x = range(len(div_df))w = 0.3ax.bar([i - w for i in x], div_df['innate_divergence'], width=w, label='Innate', color='#E74C3C', alpha=0.8)ax.bar(x, div_df['total_divergence'], width=w, label='Total', color='#3498DB', alpha=0.8)ax.bar([i + w for i in x], div_df['learned_divergence'], width=w, label='Learned', color='#2ECC71', alpha=0.8)ax.set_xticks(x)ax.set_xticklabels([f"{r['odorant']}\n({r['full_name'][:15]})" for _, r in div_df.iterrows()],                   fontsize=8, rotation=45, ha='right')ax.set_ylabel('Behavioral Divergence (Euclidean, z-scored)')ax.set_title('Concentration Sensitivity: Low → High Behavioral Change per Odorant',             fontsize=13, fontweight='bold')ax.legend()plt.tight_layout()plt.show()print("\nOdorants ranked by total behavioral divergence (low → high):")print(div_df[['odorant', 'full_name', 'total_divergence', 'innate_divergence',              'learned_divergence']].to_string(index=False))

### 3.2 NMF decomposition: latent behavioral programsNon-negative matrix factorization (NMF) decomposes the behavioral data into additive "programs" — combinations of behaviors that co-occur. We can then see how odorants shift between programs as concentration changes.

In [ ]:
# Fit NMF on all data (odorant trials only, not water)scaler = MinMaxScaler()X = scaler.fit_transform(df[behav_cols].values.astype(float))# Model selection: reconstruction error for k = 2..8ks = range(2, 9)errors = []for k in ks:    model = NMF(n_components=k, init='nndsvda', max_iter=2000, random_state=42)    model.fit(X)    errors.append(model.reconstruction_err_)fig, ax = plt.subplots(figsize=(7, 4))ax.plot(list(ks), errors, 'o-', color='steelblue', linewidth=2)ax.set_xlabel('Number of Components (k)')ax.set_ylabel('Reconstruction Error')ax.set_title('NMF Model Selection', fontsize=12, fontweight='bold')plt.tight_layout()plt.show()# Use k=3 (matches Manoel et al.'s finding of ~3 behavioral modes)K = 3print(f"\nUsing k={K} components")

In [ ]:
# Fit NMF with k=3 and inspect the basisnmf_model = NMF(n_components=K, init='nndsvda', max_iter=2000, random_state=42)W = nmf_model.fit_transform(X)H = nmf_model.components_# Basis heatmap: what behaviors define each program?fig, ax = plt.subplots(figsize=(12, 3.5))basis_df = pd.DataFrame(H, columns=behav_cols,                        index=[f'Program {i+1}' for i in range(K)])sns.heatmap(basis_df, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,            linewidths=0.5)ax.set_title('NMF Behavioral Programs (Basis Vectors)', fontsize=12, fontweight='bold')plt.tight_layout()plt.show()# Label programs based on dominant behaviorsfor i in range(K):    top_3 = basis_df.iloc[i].nlargest(3)    print(f"Program {i+1}: dominated by {', '.join(top_3.index)}")

In [ ]:
# Project all observations into NMF space, then plot concentration trajectoriesX_all = scaler.transform(df_all[behav_cols].values.astype(float))X_all = np.clip(X_all, 0, None)W_all = nmf_model.transform(X_all)# For 2D visualization, use the top 2 componentsfig, ax = plt.subplots(figsize=(10, 8))CONC_MARKERS = {'8.5uM': 'o', '850uM': 's', '85mM': 'D', 'control': '*'}cmap = plt.cm.tab20for idx, odorant in enumerate(odorants_12):    color = cmap(idx / len(odorants_12))    centroids = []        for conc in CONC_ORDER:        mask = (df['odorant'] == odorant) & (df['concentration'] == conc)        indices = df.index[mask]        # Find positions in df_all        pos = [list(df_all.index).index(i) for i in indices if i in df_all.index]        if pos:            coords = W_all[pos, :2]            ax.scatter(coords[:, 0], coords[:, 1], color=color, alpha=0.2,                      s=20, marker=CONC_MARKERS[conc])            centroids.append(coords.mean(axis=0))        # Connect centroids with arrows    if len(centroids) >= 2:        for i in range(len(centroids) - 1):            ax.annotate('', xy=centroids[i+1], xytext=centroids[i],                       arrowprops=dict(arrowstyle='->', color=color, lw=2))        ax.text(centroids[-1][0], centroids[-1][1], f' {odorant}',               fontsize=7, color=color, fontweight='bold')ax.set_xlabel(f'NMF Component 1', fontsize=11)ax.set_ylabel(f'NMF Component 2', fontsize=11)ax.set_title('Concentration Trajectories in Behavioral Space (NMF)',             fontsize=13, fontweight='bold')plt.tight_layout()plt.show()

### 🔬 SA1 — Your turn**Interpret the behavioral results:**1. Which odorants show the strongest concentration dependence? The weakest? Does this make intuitive sense given what these molecules smell like?2. Looking at the innate vs. learned divergence: do some odorants primarily shift their *defensive* behaviors with concentration, while others shift *exploratory* behaviors? What might explain this?3. The NMF programs represent latent behavioral "modes." How do odorants transition between modes as concentration increases? Are there distinct trajectories (e.g., some odorants shift from exploration → avoidance)?4. How does the innate/learned categorization map onto the NMF programs? Do the NMF components naturally separate into innate and learned modes, or is the picture more complex?

In [ ]:
# Your interpretation here

---# PART II — Concentration-Dependent Receptor Population Coding (Mainland et al.)Mainland et al. measured dose-response curves for ~300 human olfactory receptors against ~90 odorants. By fitting Hill functions to each (receptor, odorant) pair, we can predict receptor activation at *any* concentration — and ask how the population-level receptor code changes from low to high concentration.---## 4. Fit Hill curves to dose-response dataA 4-parameter Hill equation describes each receptor's response as a function of odorant concentration. The key parameter is **EC₅₀** (the concentration at half-maximal response) — this is essentially the receptor's Kd for that odorant.

In [ ]:
# Hill function and fittingdef hill_function(log_conc, baseline, top, log_ec50, hill_n):    """4-parameter Hill equation in log-concentration space."""    return baseline + (top - baseline) / (        1.0 + 10.0 ** ((log_ec50 - log_conc) * hill_n)    )def fit_sigmoid(concentrations, responses):    """Fit a Hill curve. Returns (params, r_squared) or (None, None) on failure."""    log_c = np.log10(concentrations)    y = responses    p0 = [np.percentile(y, 10), np.percentile(y, 90), np.median(log_c), 1.0]    bounds = ([-5, -5, -15, 0.1], [5, 50, 0, 10])    try:        popt, _ = curve_fit(hill_function, log_c, y, p0=p0,                           bounds=bounds, maxfev=5000, method='trf')        y_pred = hill_function(log_c, *popt)        ss_res = np.sum((y - y_pred) ** 2)        ss_tot = np.sum((y - np.mean(y)) ** 2)        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0        return popt, r2    except (RuntimeError, ValueError):        return None, None# Fit all (receptor, odorant) pairsprint("Fitting Hill curves to all (receptor, odorant) pairs...")results = []grouped = dr.groupby(["OR", "Odor"])for (or_id, odor_id), group in grouped:    conc = group["concentration"].values    resp = group["NormalizedLuc"].values    if len(conc) < 4 or len(np.unique(conc)) < 3:        continue    popt, r2 = fit_sigmoid(conc, resp)    if popt is not None:        baseline, top, log_ec50, hill_n = popt        results.append({            "OR": or_id, "Odor": odor_id,            "baseline": baseline, "top": top,            "log_ec50": log_ec50, "ec50": 10 ** log_ec50,            "hill_n": hill_n, "r_squared": r2,            "dynamic_range": top - baseline,            "gene": receptor_genes.get(or_id, f"OR{or_id}"),            "odor_name": odor_names.get(odor_id, f"Odor{odor_id}"),        })fits = pd.DataFrame(results)good_fits = fits[(fits["r_squared"] > 0.3) & (fits["dynamic_range"] > 0.3)].copy()print(f"Successfully fit {len(fits)} curves out of {grouped.ngroups} pairs")print(f"Good fits (R²>0.3, dynamic range>0.3): {len(good_fits)}")print(f"Covering {good_fits['Odor'].nunique()} odorants and {good_fits['OR'].nunique()} receptors")

### 4.1 Population delta: how much does the receptor code change with concentration?For each odorant, we compute the fractional activation of every receptor at low (10⁻⁸ M) and high (10⁻³ M) concentration — roughly matching the range tested by Manoel et al. The **population delta** is the mean change in activation across the receptor population.

In [ ]:
LOW_CONC = 1e-8    # ~8.5 µM order of magnitudeHIGH_CONC = 1e-3   # ~85 mM order of magnitudeLOG_LOW = np.log10(LOW_CONC)LOG_HIGH = np.log10(HIGH_CONC)def fractional_activation(row, log_conc):    """Fractional receptor saturation at a given concentration."""    response = hill_function(log_conc, row["baseline"], row["top"],                            row["log_ec50"], row["hill_n"])    if row["dynamic_range"] > 0:        return np.clip((response - row["baseline"]) / row["dynamic_range"], 0, 1)    return 0.0delta_records = []for odor_id, odor_group in good_fits.groupby("Odor"):    if len(odor_group) < 5:        continue    act_low = odor_group.apply(fractional_activation, axis=1, log_conc=LOG_LOW)    act_high = odor_group.apply(fractional_activation, axis=1, log_conc=LOG_HIGH)    abs_delta = np.abs(act_high - act_low)        delta_records.append({        "Odor": odor_id,        "odor_name": odor_names.get(odor_id, f"Odor{odor_id}"),        "n_good_fits": len(odor_group),        "mean_delta": abs_delta.mean(),        "n_active_low": (act_low > 0.05).sum(),        "n_active_high": (act_high > 0.05).sum(),        "n_newly_recruited": ((act_low < 0.05) & (act_high > 0.05)).sum(),        "mean_act_low": act_low.mean(),        "mean_act_high": act_high.mean(),    })deltas = pd.DataFrame(delta_records).sort_values("mean_delta", ascending=False).reset_index(drop=True)# Bar chartfig, ax = plt.subplots(figsize=(10, max(6, len(deltas) * 0.35)))colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(deltas)))ax.barh(range(len(deltas)), deltas["mean_delta"], color=colors[::-1],        edgecolor="white", linewidth=0.3)ax.set_yticks(range(len(deltas)))ax.set_yticklabels(deltas["odor_name"], fontsize=8)ax.invert_yaxis()ax.set_xlabel("Mean Per-Receptor |Δ Activation| (10⁻⁸ → 10⁻³ M)")ax.set_title("Population Delta: Receptor Code Change with Concentration",             fontsize=13, fontweight="bold")for i, row in deltas.iterrows():    ax.text(row["mean_delta"] + 0.005, i, f'n={row["n_good_fits"]}',            va="center", fontsize=7, color="gray")ax.spines["top"].set_visible(False)ax.spines["right"].set_visible(False)plt.tight_layout()plt.show()print(f"\nComputed population delta for {len(deltas)} odorants")print("\nTop 5:")print(deltas[["odor_name", "mean_delta", "n_good_fits", "n_newly_recruited"]].head().to_string(index=False))print("\nBottom 5:")print(deltas[["odor_name", "mean_delta", "n_good_fits", "n_newly_recruited"]].tail().to_string(index=False))

### 4.2 Example dose-response curvesLet's look at a high-delta and a low-delta odorant to build intuition for what population delta *means* at the single-receptor level.

In [ ]:
# Pick a high-delta and low-delta odorant with sufficient fitscandidates = deltas[deltas["n_good_fits"] >= 8]mol_high = candidates.iloc[0]mol_low = candidates.iloc[-1]fig, axes = plt.subplots(2, 4, figsize=(16, 8))for row_idx, (mol, label) in enumerate([(mol_high, "HIGH Δ"), (mol_low, "LOW Δ")]):    odor_id = mol["Odor"]    mol_fits = good_fits[good_fits["Odor"] == odor_id]    top_fits = mol_fits.nlargest(4, "dynamic_range")        for col_idx, (_, row) in enumerate(top_fits.iterrows()):        if col_idx >= 4:            break        ax = axes[row_idx, col_idx]                # Raw data        mask = (dr["OR"] == row["OR"]) & (dr["Odor"] == odor_id)        raw = dr[mask]        ax.scatter(raw["concentration"], raw["NormalizedLuc"],                  alpha=0.5, s=25, color="steelblue")                # Fitted curve        x_fit = np.logspace(-14, -1, 200)        y_fit = hill_function(np.log10(x_fit), row["baseline"], row["top"],                             row["log_ec50"], row["hill_n"])        ax.plot(x_fit, y_fit, "r-", linewidth=2)                ax.axvline(LOW_CONC, color="green", ls=":", alpha=0.7)        ax.axvline(HIGH_CONC, color="orange", ls="--", alpha=0.7)        ax.axvspan(LOW_CONC, HIGH_CONC, alpha=0.05, color="yellow")        ax.set_xscale("log")        ax.set_title(f"{row['gene']}  EC₅₀={row['ec50']:.1e}", fontsize=9)        ax.set_xlabel("Conc (M)", fontsize=8)        ax.set_ylabel("Norm. Resp.", fontsize=8)        axes[row_idx, 0].set_ylabel(f"{mol['odor_name']}\n({label})\nNorm. Resp.",                                fontsize=9, fontweight='bold')plt.suptitle("Dose-Response Curves: High vs Low Population Delta Odorants",            fontsize=13, fontweight='bold')plt.tight_layout()plt.show()

### 🔬 SA2a — Your turn**Interpret the receptor results:**1. Which odorants show the largest receptor population delta? Do they recruit many *new* receptors at high concentration, or do existing receptors simply increase their response?2. The low and high concentrations here (10⁻⁸ and 10⁻³ M) roughly match the Manoel behavioral range (8.5 µM to 85 mM). Is this a fair comparison, or are there important caveats?3. Looking at the dose-response curves: what makes a receptor "concentration-sensitive" vs. "concentration-insensitive"? (Hint: think about EC₅₀ relative to the test range.)

In [ ]:
# Your interpretation here

---# PART III — Linking Receptor Encoding to Behavioral Response (SA2)Now we test the central hypothesis: **Does the magnitude of concentration-dependent receptor code change predict the magnitude of concentration-dependent behavioral change?**We have a small number of odorants that appear in both datasets. Three of the 12 behavioral odorants have direct Mainland receptor data: **2-heptanone**, **vanillin**, and **TMT**. Beyond these, 13 odorants overlap between the broader Manoel molecular panel and Mainland.---## 5. Identify overlapping odorants

In [ ]:
# Map behavioral odorants to Mainland odorant IDs# The behavioral data uses abbreviations; Mainland uses full namesmainland_name_to_id = dict(zip(odors['OdorName'].str.lower(), odors['Odor']))mainland_name_to_name = dict(zip(odors['OdorName'].str.lower(), odors['OdorName']))overlap = []for abbr, full_name in ABBREV_TO_NAME.items():    ml_id = mainland_name_to_id.get(full_name.lower())    if ml_id is not None:        # Check if we have good fits for this odorant        n_fits = len(good_fits[good_fits['Odor'] == ml_id])        overlap.append({            'abbreviation': abbr,            'full_name': full_name,            'mainland_id': ml_id,            'n_receptor_fits': n_fits,        })overlap_df = pd.DataFrame(overlap)print("Behavioral odorants with Mainland receptor data:")print(overlap_df.to_string(index=False))print(f"\n{len(overlap_df)} of 12 behavioral odorants have receptor dose-response data")# For the broader comparison, also find all Manoel molecules in Mainlandbroader_overlap = []for _, mol_row in molecules.iterrows():    ml_id = mainland_name_to_id.get(mol_row['name'].lower())    if ml_id is not None:        n_fits = len(good_fits[good_fits['Odor'] == ml_id])        if n_fits > 0:            broader_overlap.append({                'name': mol_row['name'],                'CID': mol_row['CID'],                'mainland_id': ml_id,                'n_receptor_fits': n_fits,            })broader_df = pd.DataFrame(broader_overlap)print(f"\nBroader Manoel × Mainland overlap: {len(broader_df)} odorants with receptor fits")

## 6. Compute receptor and behavioral deltas for overlapping odorants

In [ ]:
# Receptor population delta for the overlapping behavioral odorantsreceptor_deltas = {}for _, row in overlap_df.iterrows():    ml_id = row['mainland_id']    odor_fits = good_fits[good_fits['Odor'] == ml_id]    if len(odor_fits) == 0:        continue        act_low = odor_fits.apply(fractional_activation, axis=1, log_conc=LOG_LOW)    act_high = odor_fits.apply(fractional_activation, axis=1, log_conc=LOG_HIGH)        receptor_deltas[row['abbreviation']] = {        'mean_delta': np.abs(act_high - act_low).mean(),        'n_newly_recruited': ((act_low < 0.05) & (act_high > 0.05)).sum(),        'n_fits': len(odor_fits),        'mean_act_low': act_low.mean(),        'mean_act_high': act_high.mean(),    }# Behavioral divergence for the same odorants (from SA1)behav_deltas = {}for _, row in div_df.iterrows():    behav_deltas[row['odorant']] = {        'total_divergence': row['total_divergence'],        'innate_divergence': row['innate_divergence'],        'learned_divergence': row['learned_divergence'],    }# Combine into a single dataframecombined = []for abbr in receptor_deltas:    if abbr in behav_deltas:        combined.append({            'odorant': abbr,            'full_name': ABBREV_TO_NAME.get(abbr, abbr),            'receptor_delta': receptor_deltas[abbr]['mean_delta'],            'n_newly_recruited': receptor_deltas[abbr]['n_newly_recruited'],            'n_receptor_fits': receptor_deltas[abbr]['n_fits'],            'behavioral_divergence': behav_deltas[abbr]['total_divergence'],            'innate_divergence': behav_deltas[abbr]['innate_divergence'],            'learned_divergence': behav_deltas[abbr]['learned_divergence'],        })combined_df = pd.DataFrame(combined)print("Combined receptor + behavioral data for overlapping odorants:")print(combined_df.to_string(index=False))

## 7. Correlation: receptor delta vs. behavioral divergenceThis is the core test. With only a few overlapping odorants, we obviously can't run a powerful statistical test — but we can visualize the relationship and compute a correlation. The small N is an inherent limitation that your discussion should address.

In [ ]:
if len(combined_df) >= 3:    fig, axes = plt.subplots(1, 3, figsize=(16, 5))        for ax, y_col, y_label in zip(axes,        ['behavioral_divergence', 'innate_divergence', 'learned_divergence'],        ['Total Behavioral', 'Innate/Defensive', 'Learned/Exploratory']):                x = combined_df['receptor_delta']        y = combined_df[y_col]                ax.scatter(x, y, s=100, color='steelblue', edgecolor='white', zorder=3)                # Label each point        for _, row in combined_df.iterrows():            ax.annotate(f"  {row['odorant']} ({row['full_name'][:12]})",                       (row['receptor_delta'], row[y_col]),                       fontsize=8, fontweight='bold')                # Correlation        if len(x) >= 3:            r, p = stats.pearsonr(x, y)            rho, p_rho = stats.spearmanr(x, y)            ax.set_title(f'{y_label} Divergence\nr={r:.2f} (p={p:.3f}), ρ={rho:.2f}',                        fontsize=11, fontweight='bold')                ax.set_xlabel('Receptor Population Delta', fontsize=10)        ax.set_ylabel(f'{y_label} Divergence', fontsize=10)        plt.suptitle('Receptor Code Change vs. Behavioral Change Across Concentrations',                fontsize=13, fontweight='bold', y=1.02)    plt.tight_layout()    plt.show()else:    print(f"Only {len(combined_df)} overlapping odorants — not enough for correlation plot.")    print("Consider the broader overlap analysis below instead.")

### 7.1 Broader comparison: all overlapping odorantsEven though only a few of the 12 *behavioral* odorants have receptor data, the broader Manoel molecular panel has 13 odorants overlapping with Mainland. While we don't have concentration-specific behavioral data for all of these, we can at least characterize how the receptor population delta varies across this larger set.

In [ ]:
# Population delta for all broader overlapping odorantsbroader_deltas = []for _, row in broader_df.iterrows():    odor_fits = good_fits[good_fits['Odor'] == row['mainland_id']]    if len(odor_fits) < 3:        continue        act_low = odor_fits.apply(fractional_activation, axis=1, log_conc=LOG_LOW)    act_high = odor_fits.apply(fractional_activation, axis=1, log_conc=LOG_HIGH)        broader_deltas.append({        'name': row['name'],        'mean_delta': np.abs(act_high - act_low).mean(),        'n_newly_recruited': ((act_low < 0.05) & (act_high > 0.05)).sum(),        'n_fits': len(odor_fits),        'in_behavioral_12': row['name'].lower() in [v.lower() for v in ABBREV_TO_NAME.values()],    })broader_deltas_df = pd.DataFrame(broader_deltas).sort_values('mean_delta', ascending=False)fig, ax = plt.subplots(figsize=(10, 5))colors = ['#E74C3C' if x else '#3498DB' for x in broader_deltas_df['in_behavioral_12']]ax.barh(range(len(broader_deltas_df)), broader_deltas_df['mean_delta'],        color=colors, edgecolor='white')ax.set_yticks(range(len(broader_deltas_df)))ax.set_yticklabels(broader_deltas_df['name'], fontsize=9)ax.invert_yaxis()ax.set_xlabel('Receptor Population Delta')ax.set_title('Receptor Code Change for Manoel × Mainland Overlapping Odorants',             fontsize=12, fontweight='bold')from matplotlib.patches import Patchax.legend(handles=[Patch(color='#E74C3C', label='In behavioral 12'),                   Patch(color='#3498DB', label='Broader panel only')],         loc='lower right')plt.tight_layout()plt.show()print(broader_deltas_df[['name', 'mean_delta', 'n_fits', 'in_behavioral_12']].to_string(index=False))

### 🔬 SA2b — Your turn**Interpret the cross-modal results:**1. For the overlapping odorants, is there a positive relationship between receptor delta and behavioral divergence? Even with small N, what's the direction?2. Consider the mismatch: receptor data is from *human* ORs, behavioral data is from *mice*. What are the implications? How would you frame this in your discussion?3. The concentration ranges aren't perfectly matched (Manoel uses 8.5 µM – 85 mM; we used 10⁻⁸ – 10⁻³ M for receptor modeling). How sensitive are the results to this choice? What would happen if you adjusted the receptor concentration range?4. Looking at the broader panel: do the 3 behavioral odorants span the full range of receptor deltas, or are they clustered? What does this mean for the generalizability of your findings?

In [ ]:
# Your interpretation here

---## 8. Summary figureA multi-panel figure showing the key results from each aim side by side.

In [ ]:
fig = plt.figure(figsize=(16, 10))gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.35)# Panel A: Behavioral divergence (SA1)ax_a = fig.add_subplot(gs[0, 0])div_sorted = div_df.sort_values('total_divergence')ax_a.barh(range(len(div_sorted)), div_sorted['total_divergence'], color='#3498DB')ax_a.set_yticks(range(len(div_sorted)))ax_a.set_yticklabels(div_sorted['odorant'], fontsize=7)ax_a.set_xlabel('Behavioral Divergence')ax_a.set_title('A. Behavioral Concentration\nSensitivity (SA1)', fontweight='bold', fontsize=10)# Panel B: Innate vs Learned divergence (SA1)ax_b = fig.add_subplot(gs[0, 1])ax_b.scatter(div_df['innate_divergence'], div_df['learned_divergence'],            s=60, color='#E74C3C', edgecolor='white')for _, row in div_df.iterrows():    ax_b.annotate(row['odorant'], (row['innate_divergence'], row['learned_divergence']),                 fontsize=7, textcoords='offset points', xytext=(3, 3))ax_b.plot([0, div_df[['innate_divergence', 'learned_divergence']].max().max()],         [0, div_df[['innate_divergence', 'learned_divergence']].max().max()],         'k--', alpha=0.3)ax_b.set_xlabel('Innate Divergence')ax_b.set_ylabel('Learned Divergence')ax_b.set_title('B. Innate vs Learned\nConcentration Effects (SA1)', fontweight='bold', fontsize=10)# Panel C: Receptor population delta (SA2)ax_c = fig.add_subplot(gs[0, 2])top_receptor = deltas.head(10)ax_c.barh(range(len(top_receptor)), top_receptor['mean_delta'], color='#2ECC71')ax_c.set_yticks(range(len(top_receptor)))ax_c.set_yticklabels(top_receptor['odor_name'], fontsize=7)ax_c.invert_yaxis()ax_c.set_xlabel('Receptor Population Delta')ax_c.set_title('C. Receptor Code Change\nwith Concentration (SA2)', fontweight='bold', fontsize=10)# Panel D: Cross-modal correlation (if available)ax_d = fig.add_subplot(gs[1, :2])if len(combined_df) >= 2:    ax_d.scatter(combined_df['receptor_delta'], combined_df['behavioral_divergence'],                s=120, color='#9B59B6', edgecolor='white', zorder=3)    for _, row in combined_df.iterrows():        ax_d.annotate(f"  {row['odorant']} ({row['full_name']})",                     (row['receptor_delta'], row['behavioral_divergence']),                     fontsize=9, fontweight='bold')    ax_d.set_xlabel('Receptor Population Delta (Mainland)', fontsize=11)    ax_d.set_ylabel('Behavioral Divergence (Manoel)', fontsize=11)    ax_d.set_title('D. Receptor Code Change vs Behavioral Change (SA2)',                  fontweight='bold', fontsize=10)else:    ax_d.text(0.5, 0.5, 'Insufficient overlap for\ncross-modal plot',             ha='center', va='center', fontsize=14, transform=ax_d.transAxes)    ax_d.set_title('D. Cross-Modal Comparison', fontweight='bold', fontsize=10)# Panel E: NMF programsax_e = fig.add_subplot(gs[1, 2])basis_vis = basis_df.copy()basis_vis.columns = [c[:12] for c in basis_vis.columns]sns.heatmap(basis_vis, ax=ax_e, cmap='YlOrRd', cbar=False,           annot=True, fmt='.1f', linewidths=0.5, annot_kws={'fontsize': 6})ax_e.set_title('E. NMF Behavioral\nPrograms (SA1)', fontweight='bold', fontsize=10)ax_e.tick_params(axis='x', rotation=45)plt.suptitle("Concentration-Dependent Receptor Encoding and Behavioral Response",            fontsize=14, fontweight='bold', y=1.01)plt.tight_layout()plt.show()

---## 9. Synthesis — Your turn**This is the thesis-level synthesis. Consider:**1. **Does the hypothesis hold?** Is there evidence that receptor code changes predict behavioral changes across concentrations? What's the strongest evidence for and against?2. **Innate vs. learned:** Your specific aims asked about organizing behaviors into innate vs. learned categories. Did the concentration dependence differ between these categories? Does the receptor delta predict one category better than the other?3. **Limitations to address:**   - Human receptor data vs. mouse behavior — what aspects of the comparison are still valid?   - Small N of overlapping odorants — how does this constrain your conclusions?   - Are the concentration ranges well-matched between datasets?   - NMF programs vs. innate/learned categorization — do they tell the same story?4. **What would make this study more compelling?** If you could design the ideal experiment, what would it look like?

In [ ]:
# Your synthesis here

---## 10. Next Steps- [ ] Refine the innate/learned behavioral categorization — consult ethology literature for this specific behavioral battery- [ ] Adjust the receptor concentration range to better match Manoel's 8.5 µM – 85 mM- [ ] Compute receptor delta at *three* concentrations (matching the Manoel design) to get trajectories, not just endpoints- [ ] Test whether *newly recruited* receptors (vs. increased activation of existing receptors) better predict behavioral change- [ ] Run permutation tests on the cross-modal correlation to assess significance despite small N- [ ] Consider using the broader 73-odorant Manoel panel (non-concentration data) for additional receptor overlap analyses- [ ] Explore whether specific NMF behavioral programs correspond to specific receptor activation patterns